In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("project2.ipynb")

<img src="data6.png" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

# Project 2: The Web — HTML and Web APIs

## Data 6, Fall 2026

### Due Date: Monday, November 16th at 11:59pm

In this project, you will explore text analysis on the web:
* How to use Web APIs to get data (JSON, HTML) from databases like Wikipedia and the UK's Carbon Intensity API
* How to process and analyze text with structure
* How to parse HTML with BeautifulSoup

You must submit the assignment to Gradescope. Submission instructions can be found at the bottom of this notebook. See the [syllabus](http://data6.org/fa25/syllabus/#late-policy-and-extensions) for our late submission policy.

### Project Partners

For projects, you can work with up to one partner. Please see our syllabus for our [partner collaboration policy](https://data6.org/fa25/syllabus/#projects).

If you work with a partner, please **include their name** below. **Additionally, both students are expected to submit their own notebook** even if those submissions are the same. We want you to type out your own responses to have more practice coding.

In [ ]:
partner = "NAME" # if no partner, change to "NA"; first & last name please
partner_email_address = "NAME@berkeley.edu"

# don't change the code below
print(f"For project 2, I will be working with {partner}. Their email address is {partner_email_address}")

In [ ]:
# Run this cell, don't change anything.
from datascience import *
import numpy as np

from IPython.display import HTML, display, Image
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use("ggplot")

import warnings
warnings.filterwarnings('ignore')
%reload_ext autoreload 
%autoreload 2

from bs4 import BeautifulSoup
import re
import requests

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## Wikipedia: Introduction


**Natural language processing (NLP)** is the processing of natural language information by a computer [[Wikipedia](https://en.wikipedia.org/wiki/Natural_language_processing)]. The intersection of NLP and Artificial Intelligence has evolved into the advent of AI and chatbot usage that we see today, but NLP ultimately has very humble beginnings. Many text **documents** need to be analyzed, processed, and understood by humans—in other words, **text is also data**.

Furthermore, data involving human language has structures that can't be _fully represented_ by tables (rows, columns), and other rectangular structures. Fortunately, we have the power of dictionaries and HTML to help us represent and analyze text. Let's see!

A Wikipedia article is just words...right? Not quite. Take a look at the two Wikipedia articles that we explore in this project:

* [Data Science](https://en.wikipedia.org/wiki/Data_science)
* [Python (Programming Language)](https://en.wikipedia.org/wiki/Python_(programming_language))

As you can see, there are many other features of this **text data**. There are headers, images, formatting, links, ... the list goes on. In this project, we will explore many different representations of a Wikipedia article:

1. Natural Language Processing structures:
    * One single text string of all words in the article
    * A list of all words in the article
    * A **bag of words**
1. Article HTML and document structure

**The Wikipedia API**: Wikipedia, like many other modern databases, has an API that can be used to extract all sorts of data. The [Wikimedia Action API](https://www.mediawiki.org/wiki/API:Action_API) run by WikiMedia that data scientists, software developers, and researchers use to study Wikipedia and other knowledge bases. This API does the heavy lifting for us in accessing the official API and makes it much easier for us to analyze text and HTML in our Jupyter notebooks. To use this Wikipedia API, we provide a modified URL that can handle direct data requests (as opposed to the two Wikipedia links above, which are meant for web browser requests).

We have constructed all API calls for you already, so no worries on that front—but feel free to ask us if you're curious!

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 1: Wikipedia and NLP

> What are the most important words in the Wikipedia article on [Data Science](https://en.wikipedia.org/wiki/Data_science)?

Let's answer this question using natural language processing techniques.

First, run the cell below to use the API to load the Data Science wikipedia page into `wiki_text`. (For those curious, this [API Sandbox](https://en.wikipedia.org/wiki/Special:ApiSandbox#action=query&format=json&prop=extracts&titles=Data_science&formatversion=2&exlimit=1&explaintext=1&exsectionformat=wiki)) was used to construct the URL in the request.)

In [ ]:
# just run this cell. feel free to ask us about it.

import requests
from bs4 import BeautifulSoup

headers = {'User-Agent':
           'Data 6 Student/0.0 (https://data6.org/; data6@berkeley.edu)'}

url = "https://en.wikipedia.org/w/api.php?action=query&format=json&prop=extracts&titles=Data_science&formatversion=2&exlimit=1&explaintext=1&exsectionformat=wiki"
response = requests.get(url, headers=headers)
wiki_text = response.json()["query"]["pages"][0]["extract"]
wiki_text

As we can see, `wiki_text` is a very, very long string that has lots of information but might not be very useful to process. How long is it? Well...

In [ ]:
# just run this cell
len(wiki_text)

That's over 7500 characters—but not all of them are words! These characters also include whitespace (spaces, line breaks/new lines) and punctuation. In order to understand the most common words in our Wikipedia article, we need a means of separating out words from non-words, and one word from other words.

A string where all words are jumbled together by whitespace will not help us much. Let's see if a list or an array will fare better. 

Next, run the cell below to construct `wiki_words`, a list of words in order of appearance in the article. The below cell uses [regular expressions](https://en.wikipedia.org/wiki/Regular_expression), which we don't cover in this course but are often used in many settings for text parsing. The below cell also converts all words to **lowercase**.

In [ ]:
# just run this cell
import re
def split_into_words(any_chunk_of_text):
    lowercase_text = any_chunk_of_text.lower()
    split_words = re.split("\W+", lowercase_text)
    return split_words 

wiki_words = split_into_words(wiki_text)
len(wiki_words)

There are over 1100 words in our Wikipedia article! We won't print out all of them, but verify for yourself that the first ten words in the `wiki_words` list match the first twenty words in the original `wiki_text` string:

In [ ]:
# just run this cell
wiki_words[:20]

Notice that just from the list above, we can see that the word `"scientific"` appears in our [Data Science](https://en.wikipedia.org/wiki/Data_science) article twice. Let's explore an NLP model called **bag of words** that can help us track these counts.

<hr style="border: 1px solid #fdb515;" />

## Question 1: Bag of Words

A bag of words is a very common NLP representation of text documents like Wikipedia articles. Bag of words is a simple but powerful model involving tracking **word frequencies**. A word's **frequency** in a document is the number of times it appears in a given document.

In this question, you will create a **bag of words** model as a dictionary of word frequencies:
* Each **key** is a word that appears in the document.
* Each **value** is the number of times that word appears.

<!-- BEGIN QUESTION -->

---

## Question 1a

Before we dive into our lengthy Wikipedia document, let's spend some time understanding the meaning of word frequencies.

Consider the document below (consisting of two sentences, but with all punctuation and capitalization removed):

> data 6 is data science data science is cool

Build a bag of words model for this sentence. In other words, assign `manual_word_freqs` to a dictionary of word frequencies. The keys in this dictionary should be all unique words in this sentence; the values should be the integer counts of that dictionary. 

_Notes_:
* You should **not** use a loop to create this sentence. Instead, make a new dictionary using the curly bracket syntax. 
* We have provided some keys for you, but you should supply their values, and you will need to add some more key-value pairs.
* For the purposes of this project, we assume a **word** is any lowercase alphanumeric string of characters that does not include whitespace nor punctuation. `"6"` is therefore a word; we have explicitly included it in your dictionary.

In [ ]:
manual_word_freqs = {
    "data": ...,
    "6": ..., 
    "is": ...,
    ... # add more key-value pairs here
}
manual_word_freqs

In [ ]:
grader.check("q1a")

<!-- END QUESTION -->

Now that we are equipped with this "bag of words" concept, consider what it would mean to write code to build `word_freqs`, a dictionary of word frequencies in the Wikipedia article on Data Science.

As we saw earlier, there are over 1100 words in this article. It is infeasible for us to manually count all the words. Instead, we will use the power of **iteration** to create our dictionary, one word at a time.

**Algorithm**: To build `word_freqs`, iterate over the words in `wiki_words`. For each word encountered, update its count in `word_freqs`. Of course, sometimes we will encounter a new word which does not yet exist in `word_freqs`; when we do so, we must first create an entry in the dictionary, and then increment the count.

For example, suppose `wiki_words` is the list `['apple', 'banana', 'apple']`, and we start out with an empty `word_freqs` dictionary `{}`. We iterate over the words in `wiki_words`:

1. First word: Since `'apple'` doesn't exist in the dictionary, create an entry in `wiki_words` and increment it by 1. `word_freqs` is now `{'apple': 1}`.
1. Second word: Since `'banana'` doesn't exist in the dictionary, create an entry in `wiki_words` and increment it by 1. `word_freqs` is now `{'apple': 1, 'banana': 1}`.
1. Third word: Since `'apple'` is in the dictionary, look up the entry and increment it by 1. `word_freqs` is now `{'apple': 2, 'banana': 1}`.

<!-- BEGIN QUESTION -->

---

## Question 1b

Using the previously described algorithm, complete the code below to construct `word_freqs` from `wiki_words`.

<!-- _Optional_: If you truly want to be fancy, check out the `get(key, default)` dictionary method in Python, which returns the value for `key` if `key` is in the dicitionary and `default` otherwise. See the [Python `get` documentation](https://docs.python.org/3/library/stdtypes.html#dict.get)). Otherwise, this part is totally solvable with square-bracket access of keys, provided the key is in the dictionary to begin with. Our solution doesn't use `get` -->

In [ ]:
word_freqs = {}

for word in wiki_words:
    ...

# edit the below line as needed to check your work
test_words = ["data", "science", "scientific", "and", "the", "academic"]
for word in test_words:
    print(f"{word}: {word_freqs.get(word, 'not found!')}")

In [ ]:
grader.check("q1b")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

## Question 1c: Word Frequency Table

While dictionaries can store a variety of key-value pairs, our word frequency dictionary has a very _rigid_ structure, where each (key, value) pair has the same meaning: the number of times a word appears on the Wikipedia page.

Because of this consistent structure, we can therefore **also** store our word frequencies in a **table**, with two columns: `word` and `count`, where each row is a word on the Wikipedia page.

Use your `word_freqs` dictionary to build the `word_freqs_table`, which should have the below structure:

| word | count | 
| --- | --- |
| ... | ... |
| science | 11 |
| ... | ... |

_Hint_: Use the `keys` and `values` dictionary methods and the `with_columns` Table method.

In [ ]:
word_freqs_table = Table().with_columns(
    ...
) 

word_freqs_table

In [ ]:
grader.check("q1c")

<!-- END QUESTION -->


<hr style="border: 1px solid #fdb515;" />

# Question 2: Stop words

As a reminder, our goal in this part of the project is to find the most important words in our Wikipedia article. Now that we've transformed our text data into a tabular representation—and, in particular, designed a variable that measures the "importance" of each word by count—we can use `Table` methods to process and visualize. 

However, notice that we've defined **a variable that tries to measure the "importance" of each word by its frequency**. But _some_ words, while _frequent_, do not particularly seem _"important"_ to the article content:

In [ ]:
# just run this cell
word_freqs_table.sort("count", descending=True)

As we can see, there are many words that seem important to writing the English language but are not necessarily important to the topic of Data Science: "and", "the", "of", etc. Such words are called **stop words** [[Wikipedia](https://en.wikipedia.org/wiki/Stop_word)] because processing and analyzing them is auxiliary to the purposes of understanding meaning.

There are many lists of stop words out there. Instead of constructing our own, we will use a reliable [Python package](https://pypi.org/project/stop-words/), `stop-words`:

In [ ]:
# just run this cell
!pip install stop-words

We can then use this package to create the list `stop_words`, with stop words in the English language. We've sampled out a few for you; notice that the list includes contractions and some common abbreviations in scientific text.

In [ ]:
from stop_words import get_stop_words

# Get English stop words
stop_words = get_stop_words('english')

# five random words
Table().with_columns("word", stop_words).sample(5).column("word")

---
## [Tutorial] Lists vs. Arrays: Appending Elements

To append a single element to a list, you can use the list method append. Run the next three cells below:

In [ ]:
# just run this cell
my_lst = ["some", "elements", 2]
my_lst

In [ ]:
# just run this cell
my_lst.append("in sequence")

In [ ]:
# just run this cell
my_lst

Above, the list method `append` returns nothing and directly modifies the original list. This behavior is unlike `np.append`, which returns a new array. Run the cells below for comparison:

In [ ]:
# just run this cell
arr = make_array("some", "elements")
np.append("in sequence", arr)

In [ ]:
# just run this cell
# np.append leaves arr unchanged
arr 

To "update" the original array, assign the `np.append` return value to the original array name:

In [ ]:
# just run this cell
arr = np.append("in sequence", arr)
arr

<!-- BEGIN QUESTION -->

---

## Question 2a

We can use the `stop_words` list to discern which words in a document are "meaningful" words, and which ones are stop words. Let us first try this approach on a small example. Consider the document below (all lowercase, no punctuation):

> the quick brown fox jumps over the lazy dog

In the cell below, write code that constructs `filtered_words`, a list of words in the provided sentence that are **not** stop words.

_Hints_:
* Use a `for` loop.
* `x not in seq` evaluates to `True` if `x` is not contained in the list `seq`; otherwise, it evaluates to `False`.

In [ ]:
# do not edit these lines
text = "the quick brown fox jumps over the lazy dog"
words = text.split() 

# make edits below this line
filtered_words = []
...
        
print(filtered_words)

In [ ]:
grader.check("q2a")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

## Question 2b

Use `stop_words` to filter out the rows in `word_freqs_table` that are stop words, and assign this table result to `word_freqs_nostop`. After your code runs, `word_freqs_nostop` should not have any stop words.

_Hint_: We suggest you use the `where` Table method, as opposed to looping over the table. How do you write a custom filter function?

This function would be similar to using something like `are.between`.

In [ ]:
...

word_freqs_nostop = ...
word_freqs_nostop

In [ ]:
grader.check("q2b")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<hr style="border: 1px solid #fdb515;" />

# Question 3

Finally, use `word_freqs_nostop` to visualize the frequency counts of the **top twenty most common words** on the [Data Science Wikipedia article](https://en.wikipedia.org/wiki/Data_science#), excluding stop words. Choose an appropriate visualization strategy, then write code that produces your plot.

In [ ]:
...

<!-- END QUESTION -->

**Reflection** (no response needed): Based on your diagram, what are the most important words in the Data Science Wikipedia article? Computing, statistics, and data are central—but there is also a key focus on specific domains, disciplines, and _inter_-disciplinary work. Just like this class!

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


# Part 2: Wikipedia and HTML

The NLP bag of words model we learned above is useful for analyzing semantic meaning of general articles. However, when we work with web data, we also can study the structure of the webpage itself to understand how the page categorizes and organizes its content.

In this part, we will use the BeautifulSoup package to see how key components of the [Python (programming language)](https://en.wikipedia.org/wiki/Python_(programming_language)) are organized on the Wikipedia article:

> * What are the key aspects that describe a progamming language?
> * What are built-in Python data types?
> * What are the keywords for Python statements??

Unlike before, where we just analyzed the text, we will now analyze the full HTML data from the above Wikipedia article.

Run the cell below to load the HTML of the Python Wikipedia article into the string `wiki_html`. (For those curious, this [API Sandbox](https://en.wikipedia.org/wiki/Special:ApiSandbox#action=parse&format=json&page=Python_(programming_language)&prop=text&formatversion=2) was used to construct the URL in the request.)

In [ ]:
# just run this cell

import requests
from bs4 import BeautifulSoup
headers = {'User-Agent':
           'Data 6 Student/0.0 (https://data6.org/; data6@berkeley.edu)'}

url = "https://en.wikipedia.org/w/api.php?action=parse&format=json&page=Python_(programming_language)&prop=text&formatversion=2"
response = requests.get(url,
            headers=headers)
wiki_html = response.json()["parse"]["text"]

# long HTML response, just show the first 200 chars
wiki_html[:200]

Recall that HTML is difficult to parse by itself, so we can and should use existing packages like BeautifulSoup that can effectively parse HTML structure. 

Next, load the HTML string into a BeautifulSoup document `wiki_doc`:

In [ ]:
# just run this cell
wiki_doc = BeautifulSoup(wiki_html, "html.parser")
type(wiki_doc)

Now that we have our BeautifulSoup document, let's get started!

<div class="alert alert-block alert-info">

_**Tip**_:

When completing this part, you will find it useful to keep open a separate browser window with the Wikipedia article and the HTML inspector.
* In a separate tab, navigate [to the Wikipedia article webpage](https://en.wikipedia.org/wiki/Python_(programming_language)).
* Right click (or ctrl-click), then select Inspect/Inspect Element. If you don't see this option, double-check the Data 6 lecture notes for how to adjust your browser settings.
</div>

<hr style="border: 1px solid #fdb515;" />

# Question 4: Headers

Let's tackle the question:

> What are the key aspects that describe a progamming language?

## [Tutorial] BeautifulSoup

This tutorial serves as a crash-course to BeautifulSoup and a reminder of the salient parts of the lab. But you should complete the lab before proceeding.
                                                           
While Wikipedia is not the source of all truths, it is a crowdsourced information database. Popular articles like this article have been edited by over 5000 times by hundreds or thousands of users! It is reasonable to assume that these users have **broadly agreed on how information on this page should be organized**.
For example, the users have most likely agreed on what information should be included as the headers of the Python article:

```
History
Design philosophy and features
...
External links
```

Inspect the HTML of the page and identify which HTML tag is associated with this header. 

<details>
  <summary>Click for Solution</summary>
    
The `<h2>` tag
    
</details>

Run the cell below which uses the BeautifulSoup method `find` to return the first header tag on the article page:

In [ ]:
# just run this cell
first_h2_tag = wiki_doc.find("h2")
first_h2_tag

Run the cell below which accesses the header string itself, defined as the information between the &lt;h2> and &lt;/h2> delimiters:

In [ ]:
# just run this cell
first_h2_tag.string

Note that `find` returns just the first tag. To find all header tags, we can use the `find_all` method:

In [ ]:
wiki_doc.find_all("h2")

<!-- BEGIN QUESTION -->


---

## Exercise

How do we extract the header strings from this list of tags?

In the cell below, use the `wiki_doc` BeautifulSoup document to get a list of all of the headers in the Wikipedia article. Assign this **list of strings** to `headers`.

_Hints_:
* We have provided a `for` loop structure for you. It is up to you to figure out what this loop should iterate over.See the tutorial above.
* In BeautifulSoup, you can access the contents of a tag `tag` with `tag.string`. This is not a function.
* After running your code, the first and last elements of `headers` should be `"History"` and `"External links"`, respectively. Based on our discussion above, `"External links"` should appear twice in your list.

In [ ]:
headers = []
for h2_tag in ...:
    ...

headers

In [ ]:
grader.check("q4")

<!-- END QUESTION -->


<hr style="border: 1px solid #fdb515;" />

# Question 5: Tables and Python Types

Next let's tackle the question:

> What are built-in Python data types?

The build-in data types are listed in the only table on the Wikipedia article page. 

In your HTML inspector, navigate to the singular table on the Wikipedia article page. It should be under the subheading (tag `<h3>`) "Typing."

---

## Question 5a

Before we process this table with BeautifulSoup, let's take a few moments to understand the structure of an HTML table. Answer the multiple-choice questions below. These all have hidden tests.

### Question 5a(i)

What is the `class` attribute of the `<table>` tag? Assign `table_class` to the appropriate choice number.

1. `caption`
1. `table`
1. `wikitable`

In [ ]:
table_class = ...

In [ ]:
grader.check("q5a_i")

HTML tables are organized in **row order**, meaning that the values of each row are specified before moving onto the next row. (Contrast this with the column order our datascience `Table`, where we specify values of columns across all rows, before moving onto the next column.)

---

### Question 5a(ii)
Inspect the rows of the HTML table on the Wikipedia article page.

What HTML tag is used to denote a row? Assign `row_tag` to the appropriate choice number.

1. `<tbody>`
2. `<tr>`
3. `<th>`
4. `<td>`

In [ ]:
row_tag = ...

In [ ]:
grader.check("q5a_ii")


The **header row** specifies the column labels of the HTML table. There is one header row per HTML table; all other rows are, well, rows. Note that all rows, header or not, use the same row tag you found in the previous section.

Each column label is a **header row cell**. On our page, the header row is composed of four header row cells:

```
Type
Mutability
Description
Syntax examples
```

---

### Question 5a(iii)

Inspect these header **row cells** on the Wikipedia article page, and contrast them with other row elements from non-header rows.

Do the **header row cells** have the same **tag** as other row cells? Assign `header_row_same` to `True` or `False` appropriately.

In [ ]:
header_row_same = ...

In [ ]:
grader.check("q5a_iii")

Within each row, the first row cell is a Python data type. For example, see the HTML for the first non-header row:

```
<tr>
    <td><code>bool</code></td>
    <td>immutable</td>
    <td><a href="...">Boolean value</a></td>
    <td>...</td>
</tr>
```

The text "bool" is formatted as code on the Wikipedia page; the browser knows to display this string as code because of the `<code>` tag.

---

### Question 5a(iv)

Which of the below statements about the "bool" `<code>` tag ("the tag") are true? Assign `bool_relation` to a **list** of integers of correct statements from those below. If no statements are correct, assign `bool_relation` to an empty list (e.g., `[]`).

1. The tag is a child of the first `<td>` tag.
1. The tag is a sibling to the second `<td>` tag (the one with "immutable" as a string).
1. The tag's parent is the `<tr>` row tag.
1. If the tag is a BeautifulSoup tag called `code_tag`, then we can access the string "bool" with `code_tag.string`.

This question does not have hidden tests.

In [ ]:
bool_relation = ...

In [ ]:
grader.check("q5a_iv")


---

## Question 5b

Now that we better understand HTML table structure, we can use BeautifulSoup to extract the rows of the HTML table.

Complete the code below to get the **first data row** from the HTML table. Note that the first data row is the first row **after** the header row (the header row always comes first). Your code should **return a BeautifulSoup object** (tag) associated with the first data row, i.e., an element from a `find_all` list. We have loaded in the HTML table for you as `table_tag`.

_Hints_:
* What HTML tag is used to denote a row? See your responses to the previous question.
* You can use the `find_all` method on a tag (say, `table_tag`) to find all matching tags under `table_tag`.
* The header row always comes first (but remember zero-indexing).

If your code works correctly, then running the cell below should print out the first row's HTML, similar to what we displayed to you in the previous question.

In [ ]:
# do not change this line!
table_tag = wiki_doc.find("table", "wikitable")

# edit this line
first_data_row = ...

# do not edit this line
print(first_data_row.prettify())

In [ ]:
grader.check("q5b")


---

## Question 5c

The HTML table we are analyzing is titled, "Summary of Python 3's built-in types." The first column of this table lists each Python data type, formatted to monospaced font with the `<code>` tag. We would like to extract the data types text in this first column.

In the cell below, implement the function `extract_data_type` that takes in a non-header row tag of the HTML table (`<tr>` tag) and returns the data type as a string, with no tags.

For example, passing in `first_data_row` from the previous part should return `"bool"`.

_Hint_: Consider your analysis in an earlier question. How is the `<code>` tag "descended" from the `<tr>` row tag? Is it a child? A child of a child? We've provided some suggestion of structure below. You can also directly find the `<code>` tag if you know what you're doing...!

In [ ]:
def extract_data_type(row_tag):
    first_row_cell_tag = ...
    code_tag = ...
    ...

# do not edit below this line
print(extract_data_type(first_data_row))

In [ ]:
grader.check("q5c")


---

## Question 5d

Finally, make a list `python_builtin_types` that uses the `extract_data_type` function you defined earlier to get all the Python built-in types listed in the HTML table. You may want to use a `for` loop.

If your code is implemented correctly, running the cell should print out the values in the first column of the HTML table.

In [ ]:
python_builtin_types = []

for ... in ...:
    ...

# do not edit below this line
for dtype in python_builtin_types:
    print(dtype)

In [ ]:
grader.check("q5d")

<br/><br/><br/>

Congratulations, now you're a **real** web scraping expert! Now is a good time to take a break.


<hr style="border: 1px solid #fdb515;" />

# Question 6: Subheading: "Statements and control flow"

Let's tackle the last question:

> What are the keywords for Python statements?

These keywords are located in a bulleted list under "Statements and control flow," which is a subheading under the heading "Syntax and semantics". In your separate webpage for the [Python (programming language) page](https://en.wikipedia.org/wiki/Python_(programming_language)), navigate to this area and inspect the HTML.

You will see the following HTML structure:

```
...
<div class="mw-heading mw-heading3"><h3 id="Statements_and_control_flow">Statements and control flow</h3>...</div>
<p>Python's ...</p>
<ul>
    <li>...</li>
    ...
    <li>...</li>
</ul>
<p>The assignment statement...</p>
<p>Python does not support...</p>
<div class="mw-heading mw-heading3"><h3 id="Expressions">Expressions</h3>...</div>
...
```

In this question, we would like to do the following:

a. **Get all HTML elements** pertaining to content for the "Statements and control flow" subheading.
                                                   
b. Within the bulleted list, **get all Python code text.** Code-formatted text describes Python statement keywords.

---

## Question 6a

We would like to:

> **Get all HTML elements** pertaining to content for the "Statements and control flow" subheading.

---

### Question 6a(i)

Consider the below screenshot of the Wikipedia article:

<img src='python_subheading_content.png' width='500px' alt="Python subheading screenshot">

Based on the HTML snippet above and your inspection of the HTML document, what are **all HTML elements associated with the content in the provided screenshot, including the subheading itself**?

Assign `subheader_content` to a list of appropriate numbers chosen from the below choices.

1. `<div class="mw-heading mw-heading3"><h3 id="Statements_and_control_flow">Statements and control flow</h3>...</div>`
1. `<p>Python's ...</p>`
1. `<ul><li>...</li>...<li>...</li></ul>`
1. `<p>The assignment statement...</p>`
1. `<p>Python does not support...</p>`
1. `<div class="mw-heading mw-heading3"><h3 id="Expressions">Expressions</h3>...</div>`

<!-- BEGIN QUESTION -->



In [ ]:
subheader_content = ...

In [ ]:
grader.check("q6a_i")

<!-- END QUESTION -->

Because of the structure of our document, note that all of the content that (visually, to us) looks to be "within" a subheading are actually siblings of the subheading itself, _not_ children.

See the next two cells for an explanation. Run the first cell below to get the `<div>` tag that has our "Statements and control flow" subheading:

In [ ]:
# just run this cell
statements_tag = wiki_doc.find("h3", string="Statements and control flow")
div_tag = statements_tag.find_parent()

Next, run the second cell below, which uses the `find_next_siblings` method to get all tags _after_ `div_tag`. We only print the first 100 characters of each of these tag's HTML for simplicity. But notice that very quickly, the content moves to the next subheading, "Expressions"!

In [ ]:
# just run this cell
for tag in div_tag.find_next_siblings()[:4]:
    print(str(tag)[:100], "...")
    print()

<!-- BEGIN QUESTION -->

---

## Question 6a(ii)

In HTML lists, bulleted lists are denoted by the `<ul>` tag (meaning, "unordered list"). Within this tag, each list item is denoted by `<li>`.

In the cell below, use the method `find_next_siblings` on `div_tag` to assign `list_tag` to the `<ul>` bulleted list tag. See the usage above.

If your code works, then running the below cell should display the HTML of each list item `<li>`, separated by newlines `'\n'`.

In [ ]:
list_tag = ...

# do not edit below this line
bulleted_list = list_tag.contents
bulleted_list

In [ ]:
grader.check("q6a_ii")

<!-- END QUESTION -->

If you accidentally deleted the lines above that create `bulleted_list` and your tests no longer pass, here are the lines again. Copy these lines into your cell above.

```
list_tag = ...

# do not edit below this line
bulleted_list = list_tag.contents
bulleted_list
```

<!-- BEGIN QUESTION -->

---

## Question 6b

We would like to:

> **Get all HTML elements** pertaining to content for the "Statements and control flow" subheading.

In the cell below, get all Python code statements mentioned in the list `bulleted_list`. They are marked by `code` tags:

* The first one is `=`;
* The second is `if`;
* The third is `for` (not `elif`!); 
* and so on.

_Hints_:
* What is the `code_tag`? Try `print`-ing this out by uncommenting the line.
    * Notice that some elements of `bulleted_list` are simply newlines (`"\n"`), but `code_tag` is still assigned to *something* in these "invalid" cases.
* For "valid" cases, try using `code_tag.string`. Will that give you the desired string? (Answer: Yes)

In [ ]:
py_statements = []
for tag in bulleted_list:
    code_tag = tag.find("code")
    # print(code_tag) # try uncommenting this line
    ...

# do not edit below this line
print("Python statements:")
print(py_statements)

In [ ]:
grader.check("q6b")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

## Question 6c

Read the descriptions of these Python keywords in the [Statements and Control flow section on Wikipedia](<https://en.wikipedia.org/wiki/Python_(programming_language)#Statements_and_control_flow>).
Based on these descriptions, what is one Python keyword that we haven't learned yet that you would be curious to learn more about?

Assign `unknown_keyword` to one of the keywords in `py_statements` corresponding to a keyword that we haven't learned yet. Pretty much all answers will earn credit.

In [ ]:
unknown_keyword = ...

<!-- END QUESTION -->

## Question 7

## Who is Using Social Media

Now that we have gotten a little more practice using `Beautifulsoup`, let's try and scrape some data from the Pew Research Center, a research center that focuses on answering questions based on equity in politics, science, news habits, media, among other topics. You can find out more about the [Pew Research Center here](https://www.pewresearch.org/about/). 

For the following questions, we will focus on the webpage [Americans' Use of Social Media in 2026](https://www.pewresearch.org/internet/fact-sheet/social-media/). We will do some web scraping to get specific information and move some of that data into a tabular format so we could do some additional analyses with our scraped tables.

In [ ]:
# run this code
html_str = requests.get("https://www.pewresearch.org/internet/fact-sheet/social-media/").text
html_str

pew = BeautifulSoup(html_str, "html.parser")
type(pew)

### Question 7a

There are several elements in a web page that we have covered, like `div`, `a`, `span`, `h1`, `h2`, `h3`, etc.., but there is one that will show us 

Using the `developer tools` on your web browser, look for the element that gives us only the article's information. Below are the options to choose from. 

Assign `element_location` to one of the  **string** options below. 

* `article`

* `div`

* `ul`

* `script`

In [ ]:
element_location = ...

main_page = pew.find(element_location)
main_page

In [ ]:
grader.check("q7a")

### Question 7b

Now that we have the main page's information, let's find all of the images in the fact sheet. Within `main_page`, save all of the images in `images`. 

In [ ]:
images = ...
images

In [ ]:
grader.check("q7b")

### Question 7c

We noticed that we have 4 images in our `images` list. Technically, the plot `Which social media platforms are most popular` is an image. Index within our `images` list to assign the plot to `plot_image`. 

In [ ]:
plot_image = ...

# do not alter this code
display(Image(url = plot_image['src'], width = 600, height = 500))

In [ ]:
grader.check("q7c")

### Question 7d

JP is trying to make this difficult by scraping each individual data point in the data used for the plot we created above. 

We need to find a method to show him that there is a better option than scraping the data. We have provided the parent tag of where to look with the class attribute as `wp-chart-builder-table` (You might see it as `wp-chart-builder-table active`). Within that element, there is a `div` element that has a specific class that we need to show JP that we could just export this data as a `.csv` file. 

Find the `class` that we need to show JP there is an easier way to export the data.

In [ ]:
figure_table_tabs = main_page.find_all('div', class_ = 'wp-chart-builder-table')
download_button_class = ...

specific_text = figure_table_tabs[0].find_all('div', class_ = download_button_class)[0].find('a').string.replace('\n\t\t\t\t\t', '').replace('\n\t\t\t\t', '')
specific_text

In [ ]:
grader.check("q7d")

### Question 7e

Great! You got stubborn JP to change his mind. Because he realized he was wrong, he also cleaned the exported `.csv` file so that every cell (individual value in a table) is a integer and we can examine the data. 

Read in the file `social_media_platform.csv` and assign it to `media`.

In [ ]:
media = ...
media

In [ ]:
grader.check("q7e")

### Question 7f

If you looked at the data in the table format on the website, you can see that the variable `Year` are dates that people were surveyed (more information in the *Note*). 

Instead, let's separate the months, days, and years and save them as the arrays `month`, `day`, and `yr` respectively. The values in the `media` table's `Date` column is separated as month-day-year. So for the example date of 8/5/2012, `month = 8`, `day = 5`, and `yr = 2012`. 

We'll then save these arrays as the same name `'month'`, `'day'`, and `'yr'` and create the new table `new_media`.

In [ ]:
month = make_array()
day = make_array()
yr = make_array()

for i in media.column('Date'):
  next_month = ...
  next_day = ...
  next_yr = ...
  
  month = np.append(month, int(next_month))
  day = np.append(day, int(next_day))
  yr = np.append(yr, int(next_yr))
  
new_media = ...
new_media

In [ ]:
grader.check("q7f")

### Question 7g

To better capture the data, let's get the average percentage of adults who use each social media platform for each year given that the surveys were not periodically conducted. In our new table `media_avg`, we want the averages for each social media by the year only. 

Except after some thought, let's only focus on only some social media sites. Let's iterate line plots for each of the following social media sites (`YouTube`, `LinkedIn`, `Reddit`, `Facebook`, `Instagram`, `TikTok`, and `X (formerly Twitter)`).

In [ ]:
media_avg = ...
media_avg

selected_social_media = ...

for col in selected_social_media: 
  media_avg.plot('yr', col)
  plt.title(f'Social Media: {col}')
  plt.ylim(0, 100)

In [ ]:
grader.check("q7g")

In [ ]:
# run this code
all_tables = pew.find_all('section', class_ = 'wp-block-prc-block-tab is-layout-flow wp-block-prc-block-tab-is-layout-flow')

all_tables[0].find('table').prettify()
age_table_headers_html = all_tables[0].find('table').find_all('th')

# this doesnt have a header title
print(age_table_headers_html[0])

#remove first header title by indexing the other columns or removing the first
relevant_headers = age_table_headers_html[1:len(age_table_headers_html)]

age_table_headers = make_array()
for col_name in relevant_headers:
  header = col_name.string
  age_table_headers = np.append(age_table_headers, header)
  
age_table_headers

### Question 8a

Now that we have the headers for the age table on the website, let's scrape the values within the table.

Run the code below. You'll notice that every value in the list (e.g., `<td data-prc-v-col="0" style="border-color:#fff #fff #dadbdb;border-style:dotted">YouTube</td>`) has multiple attributes for the `td` tag. For instance, if we only want the rows in the first column, we could add `attrs={'data-prc-v-col': '0'}` as a dictionary to the argument `attrs` for our `find_all` function. This will extract values in our list that matches the attribute containing `0`. 

Let's scrape all of the values within the website's age table using the `attrs` argument. In addition to scraping the values for each age range, we also need to iterate through each value in each list. Complete the code below to get the table values and iterate through each value and save it in the corresponding array for that age range.

Double check your answers against the table on the [pew research website](https://www.pewresearch.org/internet/fact-sheet/social-media/). 

In [ ]:
all_tables[0].find('table').find_all('td')

In [ ]:
age18_29_social_media = all_tables[0].find('table')...
age30_49_social_media = all_tables[0].find('table')...
age50_64_social_media = all_tables[0].find('table')...
age65_social_media = all_tables[0].find('table')...

age18_29_row_array = make_array()
age30_49_row_array = make_array()
age50_64_row_array = make_array()
age65_row_array = make_array()

for row in np.arange(len(age18_29_social_media)):
  age18_29_row = ...
  age30_49_row = ...
  age50_64_row = ...
  age65_row = ...
  
  age18_29_row_array = np.append(age18_29_row_array, int(age18_29_row))
  age30_49_row_array = np.append(age30_49_row_array, int(age30_49_row))
  age50_64_row_array = np.append(age50_64_row_array, int(age50_64_row))
  age65_row_array = np.append(age65_row_array, int(age65_row))

social_media_names = all_tables[0].find('table')...

social_media_row_array = make_array()
for social in social_media_names:
  social_media = social.string
  social_media_row_array = np.append(social_media_row_array, social_media)

age_ds_table = Table().with_columns(
  'Social Media', social_media_row_array,
  age_table_headers[0], age18_29_row_array,
  age_table_headers[1], age30_49_row_array,
  age_table_headers[2], age50_64_row_array,
  age_table_headers[3], age65_row_array
)
...
age_ds_table

In [ ]:
grader.check("q8a")

## Part 3: Using an API (without keys)

### Question 9

To get more practice with using dictionaries, let's use an application programming interface or an API. We can think of an API as the bridge between us and some data, where we use the API to make calls to get data for us. This is different from web scraping because APIs are set up for us to pull data rather than having to go through a website, find data, and scrape it ourselves.

For the following problems, we are going to use the Official Carbon Intensity API for Great Britain developed by National Energy System Operator (NESO).

[You can find documentation information about the API here](https://carbon-intensity.github.io/api-definitions/?python#carbon-intensity-api-v2-0-0), including instructions on how to call the API. We are using this API because it does not require us to sign up for anything and obtain an API key. An API key is a password that is unique to each user to use the API. 

[If you want to know more about the Official Carbon Intensity API here](https://carbonintensity.org.uk/).

Be aware that because we don't need an API key some of the Python specific code may not be 100% correct. 

### Question 9a

Let's grab data from October 2026. To keep the amount of data to a modest amount, we are going to start from October 1st and get every 7 days from the 1st. In the code below, we will have a list of 5 different dates in the `dates` list.

For the actual API call, we will be using [the intensity data found here](https://carbon-intensity.github.io/api-definitions/?python#get-intensity-from-to). This will show us how to organize our API call to choose the correct date for `from` and the date for `to`. 

For this question, we will not iterate through the dates. Instead use the first and second value in the `dates` list by indexing them using square bracket notation. The `.json()` changes the API call to a dictionary.

*Hint*: Make sure you pay attention to what kind of brackets are surrounding the data. Specifically, look for `[` and `{` to know whether there is a list (`[]`) or a dictionary (`{}`).

In [ ]:
api_headers = {
  'Accept': 'application/json'
}

dates = []
for ...
  single_date = f'2026-10-{day}'
  dates.append(single_date)

intensity = requests.get(f'https://api.carbonintensity.org.uk/intensity/{...}/{...}', params={}, headers = api_headers)

intensity_data = intensity.json()...
...
intensity_data

In [ ]:
grader.check("q9a")

### Question 9b

Now that we have the JSON data assigned to `intensity_data`, we could use square bracket notation and indexing to get the values in each dictionary. If we wanted to use the data below to make a table with, it would probably be best to create a function that can call each `key` in our dictionaries. 

```
{'from': '2026-09-30T23:30Z',
  'to': '2026-10-01T00:00Z',
  'intensity': {'forecast': 157, 'actual': 179, 'index': 'high'}}
```

Create a function to get the `values` for each `key`. We have included the function name `find_values_in_dict`, as well as the arguments `specific_dict` for the specific dictionary since we have a week of data, `key` for each key in the first dictionary, and `sub_key` for the dictionary that is within each dictionary. 

You'll also notice that there are values after each argument in our function, such as `specific_dict: dict`. These are called **type annotations** and are hints for what each argument should be. So `specific_dict` needs to be a dictionary, `key` is a string, and `sub_key` is also a string. Additionally, we have set `sub_key` with a default value of `None`. This is because our function will not always require a `sub_key` argument, so as a default we can keep it as `None`.

In [ ]:
def find_values_in_dict(specific_dict: dict, key: str, sub_key: str = None):
    if ...:
        intense_dict = ...
        return intense_dict[sub_key]
    else:
        return ...
...
# run this code to get a from date
print(find_values_in_dict(intensity_data[0], 'from'))
# run this code to get the category of carbon intensity
print(find_values_in_dict(intensity_data[0], 'intensity', 'index'))

In [ ]:
grader.check("q9b")

### Question 9c

With our function `find_values_in_dict`, we can now iterate through each data collection time point. Iterate through the list `intensity_data` to get the value for each key in each of the dictionaries we have saved in `intensity_data`.

```
{'from': '2026-09-30T23:30Z',
  'to': '2026-10-01T00:00Z',
  'intensity': {'forecast': 157, 'actual': 179, 'index': 'high'}}
```

In [ ]:
from_30min_values = make_array()
to_30min_values = make_array()
forecast_30min_values = make_array()
actual_30min_values = make_array()
cat_30min_values = make_array()

for each_dict in intensity_data:
  min30_froms =...
  min30_tos =...
  min30_forecasts =...
  min30_actuals =...
  min30_cats =...
  
  from_30min_values = np.append(from_30min_values, min30_froms)
  to_30min_values = np.append(to_30min_values, min30_tos)
  forecast_30min_values = np.append(forecast_30min_values, min30_forecasts)
  actual_30min_values = np.append(actual_30min_values, min30_actuals)
  cat_30min_values = np.append(cat_30min_values, min30_cats)
  
min30_tbl = Table().with_columns(
  'from', from_30min_values,
  'to', to_30min_values,
  'forecast', forecast_30min_values,
  'actual', actual_30min_values,
  'category', cat_30min_values
)
...
min30_tbl

In [ ]:
grader.check("q9c")

### Question 9d

The `forecast` column is the predicted carbon intensity while the `actual` column is the carbon intensity that was actually measured. While we have usually viewed data over time for our variable of interest, let's plot the difference between the `actual` column and the `forecast` column. Since we have two dates, let's use the `from` column to plot the difference over time. 

In [ ]:
diff = ...

diff_tbl = min30_tbl.with_column('Difference', diff)

diff_tbl.plot('from', 'Difference')

In [ ]:
grader.check("q9d")

### Question 9e

Finally, let's iterate through each pair of dates. The issue is that if we use the `dates` list then we'll just iterate through the same date. Instead, we have to create a list of the first 4 dates and another list with the last 4 dates. We'll call the list with the first 4 dates `start_dates` and the other list `end_dates`. Create the lists using square bracket notation.

In [ ]:
start_dates = ...
end_dates = ...

stat_data = []
for start, end in zip(start_dates, end_dates):
  stat_weeks = requests.get(f'https://api.carbonintensity.org.uk/intensity/stats/{start}/{end}', params={}, headers = api_headers)
  stat_data.append(stat_weeks.json()['data'])

from_values = make_array()
to_values = make_array()
min_values = make_array()
max_values = make_array()
avg_values = make_array()
cat_values = make_array()

for i in np.arange(len(stat_data)):
  froms = find_values_in_dict(stat_data[i][0], 'from')
  tos = find_values_in_dict(stat_data[i][0], 'to')
  mins = find_values_in_dict(stat_data[i][0], 'intensity', 'min')
  maxs = find_values_in_dict(stat_data[i][0], 'intensity', 'max')
  avgs = find_values_in_dict(stat_data[i][0], 'intensity', 'average')
  cats = find_values_in_dict(stat_data[i][0], 'intensity', 'index')
  
  from_values = np.append(from_values, froms)
  to_values = np.append(to_values, tos)
  min_values = np.append(min_values, mins)
  max_values = np.append(max_values, maxs)
  avg_values = np.append(avg_values, avgs)
  cat_values = np.append(cat_values, cats)
  
intensity_tbl = Table().with_columns(
  'from', from_values,
  'to', to_values,
  'min', min_values,
  'max', max_values,
  'average', avg_values,
  'category', cat_values
)

intensity_tbl

In [ ]:
grader.check("q9e")

# Done!!!

We hope you have learned a lot from this project! Web scraping and crawling is a huge part of the internet today, and now you have learned that it's not much more than strings and HTML soup that, sometimes, is not so beautiful. Congratulations!!!

## Pets of Data 6

Obi is beat after submitting that project. At least they got a good amount of practice creating functions, using dictionaries, beautifulsoup, and learning how to use publicly avaiable APIs.

<img src="obi.png" width="50%" alt="White dog in dog bed"/>

## Submission

Below, you will see two cells. Running the first cell will automatically generate a PDF of all questions that need to be manually graded, and running the second cell will automatically generate a zip with your autograded answers. You are responsible for submitting both the coding portion (the zip) and the written portion (the PDF) to their respective Gradescope portals. **Please save before exporting!**

> **Important: You must correctly assign the pages of your PDF after you submit to the correct gradescope assignment. If your pages are not correctly assigned and/or not in the correct PDF format by the deadline, we reserve the right to award no points for your written work.**

If there are issues with automatically generating the PDF in the first cell, you can try downloading the notebook as a PDF by clicking on `File -> Save and Export Notebook As... -> PDF`. If that doesn't work either, you can manually take screenshots of your answers to the manually graded questions and submit those. Either way, **you are responsible for ensuring your submission follows our requirements, we will NOT be granting regrade requests for submissions that don't follow instructions.**

> **TO AVOID AUTOGRADER ERRORS: Before running the cells below, _SAVE_ your work (either with the keyboard shortcut `CMD+S` for Macs, or `Ctrl+S` for Windows, or by clicking on the floppy disk in the top left) and click `Run -> Run All Cells` in the toolbar.**

In [ ]:
from otter.export import export_notebook
from os import path
from IPython.display import display, HTML
name = 'project02'
export_notebook(f"{name}.ipynb", filtering=True, pagebreaks=True, exporter_type = "html")
if(path.exists(f'{name}.pdf')):
    display(HTML(f"Download your PDF <a href='{name}.pdf' download>here</a>."))
else:
    print("\n Pdf generation failed, please try the other methods described.")

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)